# 12 — Spatial diagnostics: water mask, radius, pixel maps

**Scientific question.** How sensitive is the SWOT estimate to the water-mask and
aggregation-radius choices, and where do the accepted pixels lie?

**Outputs** `Fig09_*`, `Fig10_*`, `Fig11_*`, `Fig14_*`.

In [1]:
# --- provenance: environment, git SHA and input hashes -----------------------
import hashlib, json, subprocess, sys, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd, geopandas as gpd, pyproj, rasterio, matplotlib
from swot_dnipro import config as CFG

np.random.seed(CFG.SEED)

def sha256(p, cap=64 * 1024 * 1024):
    p = Path(p)
    if not p.exists() or p.stat().st_size > cap:
        return f"(skipped: {'missing' if not p.exists() else 'too large'})"
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()[:16]

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       cwd=CFG.ROOT, text=True).strip()
    except Exception:
        return "(not a git checkout)"

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("geopandas   ", gpd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("pyproj/PROJ ", pyproj.__version__, "/", pyproj.proj_version_str)
print("rasterio/GDAL", rasterio.__version__, "/", rasterio.__gdal_version__)
print("git SHA     ", git_sha())
print("seed        ", CFG.SEED)
print()
for p in [CFG.EGG2015_TIF, CFG.UA2019Z_ASC, CFG.CORRECTOR_BY_STATION,
          CFG.KHERSON_GAUGE_PARQUET, CFG.KHERSON_ATL13]:
    print(f"  {sha256(p)}  {Path(p).name}")


python       3.12.3
numpy        2.5.2
pandas       3.0.5
geopandas    1.1.4
matplotlib   3.11.1
pyproj/PROJ  3.7.2 / 9.5.1
rasterio/GDAL 1.5.1 / 3.12.4
git SHA      7d9580b
seed         42

  (skipped: too large)  egg_2015.tif
  297440243e2ed8d6  ua_2019z.asc
  92d9fd71a1c8da39  egg2015_to_evrf2019_by_station.csv
  3380b29a0d68016b  80805_yearbook.parquet
  1deb7b5cfd362bd7  kherson_atl13_pass_levels.parquet


In [2]:
wm = pd.read_csv(CFG.FIGDATA / "Fig09_water_mask_sensitivity.csv")
rad = pd.read_csv(CFG.FIGDATA / "Fig10_radius_sensitivity.csv")
display(wm); display(rad)
print("Accepted: classification == 4 (open_water), radius 1 km.")
print("Broader classes move the median by up to "
      f"{(wm.median_residual_m.max() - wm.median_residual_m.min()):.3f} m;")
print("radius 0.5 -> 10 km moves it by "
      f"{(rad.median_residual_m.iloc[-1] - rad.median_residual_m.iloc[0]):+.3f} m "
      "(real channel gradient).")

,strategy,classes,description,median_residual_m,nmad_residual_m,median_n_px,total_n_px,median_H_m,median_within_date_nmad_m,n_dates,accepted
0,4,4,open_water only (ACCEPTED),0.015068,0.054150,1723.0,10344,0.515910,0.122695,6,True
1,3+4,3+4,+ water_near_land,0.039531,0.049522,2189.5,12979,0.538074,0.162057,6,False
2,4+7,4+7,+ open_low_coh_water,0.025132,0.055639,1976.0,12289,0.537301,0.165490,6,False
3,3+4+6+7,3+4+6+7,+ all low-coherence,0.099733,0.033024,2876.0,17532,0.607021,0.358305,6,False
4,3+4+5+6+7,3+4+5+6+7,+ dark water,0.099569,0.036302,2935.0,17692,0.607021,0.360837,6,False


,radius_km,median_residual_m,nmad_residual_m,median_H_m,median_n_px,n_dates
0,0.5,0.052899,0.050530,0.560688,524.5,6
1,1.0,0.015068,0.054150,0.515910,1723.0,6
2,2.0,-0.001746,0.043227,0.494276,5326.0,6
3,3.0,-0.009669,0.029808,0.495177,12535.0,6
4,5.0,-0.029812,0.031580,0.477321,45577.0,6
5,10.0,-0.034576,0.034921,0.473115,123693.5,6


Accepted: classification == 4 (open_water), radius 1 km.
Broader classes move the median by up to 0.085 m;
radius 0.5 -> 10 km moves it by -0.087 m (real channel gradient).
